# 📖 Tutorial 03: Monte Carlo Tree Search (MCTS)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lessen-xu/Hybrid-Chess/blob/main/notebooks/03_mcts_explained.ipynb)

---

## Who is this for?

You've seen Alpha-Beta search in Tutorial 02. It works well but requires a hand-crafted evaluation function. This tutorial introduces **Monte Carlo Tree Search (MCTS)** — the algorithm at the heart of AlphaZero.

**What you'll learn:**
1. Why Alpha-Beta struggles with large branching factors
2. The 4 phases of MCTS: **Select → Expand → Simulate → Backpropagate**
3. The **UCB1** formula: balancing exploration vs exploitation
4. How to build a working MCTS from scratch
5. How AlphaZero transforms MCTS by replacing random rollouts with a **neural network**

**Prerequisites:** Tutorial 02 (game trees, evaluation), basic probability.

---

## 0. Setup

In [ ]:
# Uncomment for Google Colab:
# !pip install -q git+https://github.com/lessen-xu/Hybrid-Chess.git

In [ ]:
import math, random, time
from collections import defaultdict
import numpy as np
from hybrid.core.env import HybridChessEnv, GameState
from hybrid.core.types import Side, Move
from hybrid.core.rules import apply_move, generate_legal_moves, terminal_info, TerminalStatus
from hybrid.core.render import render_board
from hybrid.core.config import MAX_PLIES
print("✅ Ready")

---

## 1. From Alpha-Beta to MCTS

### 1.1 Why Alpha-Beta struggles

Alpha-Beta search works beautifully for games with manageable branching factors:
- Chess: ~35 legal moves per position → depth 10+ is feasible with optimizations
- Go: ~250 legal moves → depth 2 already requires ~62,500 evaluations
- Hybrid Chess: ~30–50 legal moves, similar to Chess

But there's a deeper problem: Alpha-Beta requires a **good evaluation function** at every leaf node. Writing one is hard — especially for novel games like Hybrid Chess where no human expert knowledge exists.

### 1.2 The Monte Carlo idea

What if, instead of evaluating a position with heuristics, we **simulated random games** from that position and counted how many times each side won?

- Play 100 random games from position A → Chess wins 60 times → A ≈ +0.2 for Chess
- Play 100 random games from position B → Chess wins 30 times → B ≈ −0.4 for Chess

This is the **Monte Carlo method**: use random sampling to estimate a value. No hand-crafted heuristics needed.

### 1.3 The key insight of MCTS

Pure Monte Carlo (simulate random games from every child) wastes effort on obviously bad moves. MCTS is smarter — it **focuses** simulations on the most promising branches:

> MCTS builds a search tree incrementally, simulation by simulation, spending more time on moves that look promising while still occasionally exploring new ones.

This is the **exploration vs exploitation** tradeoff — one of the most fundamental ideas in all of AI.

---

## 2. The Four Phases of MCTS

Each MCTS simulation follows four steps:

```
    ┌─────────────────────────────────────────────────────────────────────┐
    │                     One MCTS Simulation                            │
    │                                                                     │
    │  1. SELECT          2. EXPAND         3. SIMULATE    4. BACKPROP   │
    │                                                                     │
    │  Walk down the     Add one new       Play random     Update win/   │
    │  tree using        child node        moves until     visit counts  │
    │  UCB1 scores       to the tree       game ends       back to root  │
    │                                                                     │
    │      ●                  ●                  ●              ●         │
    │     / \                / \                 │           ↑ 10/15      │
    │    ●   ●   →         ●   ●   →       random  →       / \          │
    │   / \               / \   \          playout        5/8  5/7       │
    │  ●   ●             ●   ●  [NEW]       │            / \   \        │
    │                                      Win/Loss     3/5 2/3 [+1]    │
    └─────────────────────────────────────────────────────────────────────┘
```

### Phase 1: Selection

Starting from the root, walk down the tree, at each node choosing the child with the highest **UCB1 score** (see Section 3). Continue until you reach a node that hasn't been fully explored.

### Phase 2: Expansion

Add **one new child node** to the tree (representing an untried move from the selected node).

### Phase 3: Simulation (Rollout)

From the new node, play a **random game** until it ends. This is called a "rollout" or "playout". The result (win/loss/draw) tells us roughly how good this position is.

### Phase 4: Backpropagation

Pass the result back up the tree, updating **visit counts** and **win counts** at every node along the path.

After many simulations (e.g., 100–800), the root's children have been visited many times. We pick the child with the **most visits** as our move.

---

## 3. UCB1: Balancing Exploration vs Exploitation

### 3.1 Theory: The multi-armed bandit problem

Imagine you're in a casino with multiple slot machines. Each has a different (unknown) payout rate. You have limited pulls. How do you maximize your winnings?

- **Exploitation**: Always play the machine that's paid out the most so far
- **Exploration**: Try machines you haven't played much — they might be even better

The optimal strategy balances both. This is exactly the problem MCTS faces when choosing which child to visit.

### 3.2 The UCB1 formula

The **Upper Confidence Bound 1** formula provides a mathematically principled solution:

$$\text{UCB1}(i) = \underbrace{\frac{w_i}{n_i}}_{\text{exploitation}} + C \cdot \underbrace{\sqrt{\frac{\ln N}{n_i}}}_{\text{exploration}}$$

Where:
- $w_i$ = wins from child $i$
- $n_i$ = visits to child $i$
- $N$ = visits to the parent node
- $C$ = exploration constant (typically $\sqrt{2}$)

**Intuition:**
- The **first term** (exploitation) favors nodes that have high win rates
- The **second term** (exploration) favors nodes that haven't been visited much (the $n_i$ in the denominator makes rarely-visited nodes attractive)
- As $n_i$ grows, the exploration bonus shrinks → we settle on the best option
- The **constant C** controls the explore/exploit tradeoff: larger C = more exploration

### 3.3 Why UCB1 is remarkable

UCB1 has a **provable guarantee**: the regret (how much worse than optimal you do) grows only logarithmically with the number of trials. In plain English: you'll find the best option quickly, and you won't waste too many trials on bad options.

---

## 4. Building MCTS from Scratch

### 4.1 The Node class

Each node in the MCTS tree stores:
- The game state
- Visit count (`n`) and total value (`w`)
- Pointers to parent and children
- List of untried moves

In [ ]:
class MCTSNode:
    """A node in the MCTS search tree."""
    
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move                # the move that led to this node
        self.children = []              # list of MCTSNode
        self.n = 0                      # visit count
        self.w = 0.0                    # total value (from this node's side perspective)
        
        # Untried moves = legal moves that don't have a child node yet
        self.untried_moves = generate_legal_moves(state.board, state.side_to_move)
    
    @property
    def q(self):
        """Average value (win rate)."""
        return self.w / self.n if self.n > 0 else 0.0
    
    def ucb1(self, c=1.41):
        """UCB1 score for selection."""
        if self.n == 0:
            return float('inf')  # unvisited → always explore
        exploitation = self.q
        exploration = c * math.sqrt(math.log(self.parent.n) / self.n)
        return exploitation + exploration
    
    def is_fully_expanded(self):
        return len(self.untried_moves) == 0
    
    def is_terminal(self):
        return len(self.untried_moves) == 0 and len(self.children) == 0

print("✅ MCTSNode class defined")

### 4.2 The MCTS algorithm

Now let's implement the 4 phases:

In [ ]:
def mcts_select(node):
    """Phase 1: Walk down tree using UCB1 until a non-fully-expanded node."""
    while node.is_fully_expanded() and node.children:
        node = max(node.children, key=lambda c: c.ucb1())
    return node


def mcts_expand(node):
    """Phase 2: Add one new child by trying an untried move."""
    if not node.untried_moves:
        return node
    move = node.untried_moves.pop()  # take one untried move
    new_board = apply_move(node.state.board, move)
    child_state = GameState(
        board=new_board,
        side_to_move=node.state.side_to_move.opponent(),
        ply=node.state.ply + 1,
        repetition=node.state.repetition,
    )
    child = MCTSNode(child_state, parent=node, move=move)
    node.children.append(child)
    return child


def mcts_simulate(state, max_depth=80):
    """Phase 3: Random rollout until game ends or max_depth."""
    current = state
    for _ in range(max_depth):
        info = terminal_info(current.board, current.side_to_move,
                            current.repetition, current.ply, MAX_PLIES)
        if info.status != TerminalStatus.ONGOING:
            if info.winner == state.side_to_move:
                return 1.0    # win for the node's side
            elif info.winner is not None:
                return -1.0   # loss
            return 0.0        # draw
        
        moves = generate_legal_moves(current.board, current.side_to_move)
        if not moves:
            return 0.0
        move = random.choice(moves)
        new_board = apply_move(current.board, move)
        current = GameState(
            board=new_board,
            side_to_move=current.side_to_move.opponent(),
            ply=current.ply + 1,
            repetition=current.repetition,
        )
    return 0.0  # reached max depth → treat as draw


def mcts_backpropagate(node, value):
    """Phase 4: Update n and w back to root."""
    while node is not None:
        node.n += 1
        # Value alternates sign because sides alternate
        node.w += value
        value = -value  # opponent's perspective
        node = node.parent


def mcts_search(state, n_simulations=100):
    """Run MCTS and return the best move."""
    root = MCTSNode(state)
    
    for _ in range(n_simulations):
        # 1. Select
        node = mcts_select(root)
        # 2. Expand
        if node.untried_moves:
            node = mcts_expand(node)
        # 3. Simulate
        value = mcts_simulate(node.state)
        # 4. Backpropagate
        mcts_backpropagate(node, value)
    
    return root

print("✅ MCTS functions defined")

In [ ]:
# Run MCTS from the opening position
env = HybridChessEnv()
state = env.reset()

print("Running MCTS with 200 simulations...")
t0 = time.time()
root = mcts_search(state, n_simulations=200)
dt = time.time() - t0

print(f"Done in {dt:.2f}s")
print(f"Root visited {root.n} times")
print(f"\nTop 10 moves by visit count:")
print(f"{'Move':<20} {'Visits':>7} {'Win Rate':>9} {'UCB1':>8}")
print("─" * 48)

children_sorted = sorted(root.children, key=lambda c: c.n, reverse=True)
for child in children_sorted[:10]:
    mv = child.move
    piece = state.board.get(mv.fx, mv.fy)
    move_str = f"{piece.kind.name} ({mv.fx},{mv.fy})→({mv.tx},{mv.ty})"
    wr = child.q
    print(f"{move_str:<20} {child.n:>7} {wr:>+8.3f}  {child.ucb1():>7.3f}")

best = children_sorted[0]
print(f"\n🎯 Best move: {state.board.get(best.move.fx, best.move.fy).kind.name} "
      f"({best.move.fx},{best.move.fy})→({best.move.tx},{best.move.ty}) "
      f"(visited {best.n} times, win rate {best.q:+.3f})")

### 4.3 Visualizing the search tree

Let's see how MCTS distributes its simulations across the tree:

In [ ]:
# Distribution of visits across root children
visits = [c.n for c in root.children]
total = sum(visits)

print(f"Total children: {len(root.children)}")
print(f"Total simulations: {total}")
print(f"\nVisit distribution (each █ ≈ 2 visits):")
print()

for child in children_sorted[:15]:
    mv = child.move
    piece = state.board.get(mv.fx, mv.fy)
    bar = '█' * (child.n // 2)
    label = f"{piece.kind.name[:5]:>5}"
    print(f"  {label} │{bar} {child.n}")

if len(root.children) > 15:
    rest = sum(c.n for c in children_sorted[15:])
    print(f"  other │{'░' * (rest // 2)} {rest}")

print(f"\n💡 MCTS focuses most simulations on a few promising moves.")
print(f"   This is the exploration/exploitation tradeoff in action!")

---

## 5. From Pure MCTS to AlphaZero MCTS

### 5.1 The problem with random rollouts

Our MCTS above uses **random rollouts** to estimate position values. This works but has problems:

- **Noisy**: Random games are very different from real games. A position might be winning but random play loses it 50% of the time.
- **Slow**: We need many simulations to get stable estimates.
- **No prior knowledge**: MCTS explores all moves roughly equally at first.

### 5.2 AlphaZero's two key innovations

AlphaZero replaces the random rollout with a **neural network** that provides:

1. **Value** $v(s)$ — "How good is this position?" (replaces the random rollout)
2. **Policy** $\pi(a|s)$ — "Which moves are most promising?" (guides the search)

```
    Classical MCTS                  AlphaZero MCTS
    ─────────────                   ──────────────
    Select:   UCB1                  Select:   PUCT (uses policy prior)
    Expand:   random child          Expand:   same, but with policy priors
    Simulate: RANDOM ROLLOUT        Simulate: NEURAL NETWORK v(s)
    Backprop: same                  Backprop: same
```

### 5.3 The PUCT formula

AlphaZero replaces UCB1 with **PUCT** (Predictor + Upper Confidence Bound for Trees):

$$\text{PUCT}(s, a) = \underbrace{Q(s,a)}_{\text{value estimate}} + c_{\text{puct}} \cdot \underbrace{P(s,a)}_{\text{policy prior}} \cdot \frac{\sqrt{N(s)}}{1 + N(s,a)}$$

Where:
- $Q(s,a)$ = mean value of action $a$ from state $s$ (from backpropagated NN evaluations)
- $P(s,a)$ = the neural network's **prior probability** for action $a$ — this is the policy output
- $N(s)$ = visit count of the parent
- $N(s,a)$ = visit count of this child
- $c_{\text{puct}}$ = exploration constant

The key difference from UCB1: the **policy prior** $P(s,a)$ replaces the uniform exploration bonus. The neural network tells MCTS which moves are worth exploring, so it doesn't waste simulations on obviously bad moves.

### 5.4 The virtuous cycle

This creates a powerful feedback loop:

```
   Better Network  →  Better MCTS (more accurate value, smarter policy)
        ↑                           ↓
   Training on         →  Better Training Data (from self-play w/ MCTS)
   Self-Play Data
```

1. MCTS uses the network to evaluate positions and guide search
2. The MCTS visit distribution $\pi$ is a **better policy** than the raw network output (because search refines it)
3. We train the network to match $\pi$ (the improved policy) and predict game outcomes $z$
4. The improved network makes MCTS even better → repeat

This is the **core mechanism** of AlphaZero. We'll see it in action in Tutorial 04.

In [ ]:
# Summary comparison: Random rollouts vs Neural Network evaluation

print("Comparison: Classical MCTS vs AlphaZero MCTS")
print()
print(f"{'Aspect':<25} {'Classical MCTS':<25} {'AlphaZero MCTS'}")
print("─" * 75)
comparisons = [
    ("Value estimation",     "Random rollouts",        "Neural network v(s)"),
    ("Search guidance",      "Uniform (UCB1)",         "Policy prior P(s,a) (PUCT)"),
    ("Simulations needed",   "1000s for stable est.",  "100-800 is enough"),
    ("Domain knowledge",     "None required",          "Learned from self-play"),
    ("Improves over time?",  "No",                     "Yes (training loop)"),
    ("Quality of estimates", "Noisy (random play)",    "Smooth (learned eval)"),
]
for aspect, classical, az in comparisons:
    print(f"{aspect:<25} {classical:<25} {az}")

print(f"\n💡 AlphaZero MCTS is essentially the same algorithm, but with")
print(f"   a neural network providing better value estimates and search guidance.")
print(f"   The training process (Tutorial 04) is what makes the network good.")

---

## 🎓 Summary

| Concept | What we learned |
|---------|----------------|
| **Monte Carlo method** | Estimate values by random sampling |
| **MCTS 4 phases** | Select (UCB1) → Expand → Simulate (rollout) → Backpropagate |
| **UCB1 formula** | Balance exploitation (high win rate) and exploration (low visit count) |
| **Exploration constant C** | Controls explore/exploit tradeoff; C=√2 is standard |
| **Visit count = confidence** | More visits → more reliable value estimate |
| **AlphaZero PUCT** | Replaces UCB1; uses neural network policy prior to guide search |
| **Neural network replaces rollout** | v(s) gives instant, smooth value estimates |
| **The virtuous cycle** | Better network → better MCTS → better data → better network |

## ⏭️ What's Next?

- **Tutorial 04: AlphaZero Training** — Build the neural network, generate self-play data, train the policy and value heads
- **Tutorial 05: Experiments** — Reward shaping, variant ablation, custom architectures